# Figure 11: $\delta A$ injection comparison

Input: `data/fig11_tgr_draws.npz`. It stores the `quadratic_tgr` posterior draws for GW250114, the zero-noise injection, and 50 real-noise GR injections. These are the exact PyCBC `read_samples` draws produced with `numpy.random.seed(42)`. The prior is uniform on $[-5,1]$ and $A=A_\mathrm{GR}(1-\delta A)$.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

fig_width = 3 * 246.0 / 72.27
fig_height = fig_width * (np.sqrt(5) - 1.0) / 2.0
mpl.rcParams.update({
    'axes.labelsize': 24,
    'font.family': 'serif',
    'font.serif': 'Computer Modern Roman',
    'font.size': 24,
    'legend.fontsize': 20,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'axes.grid': True,
    'text.usetex': True,
    'savefig.dpi': 100,
    'lines.markersize': 14,
    'figure.figsize': [fig_width, fig_height],
})

from scipy.stats import gaussian_kde

data = np.load(ROOT / 'data/fig11_tgr_draws.npz')
production = data['prod']
zero_noise = data['zero30k']
injections = [data[f'inj{index:02d}'] for index in range(50)]

median_production = np.median(production)
quantile_production = np.mean(production > 0)
median_injections = np.array([np.median(samples) for samples in injections])
quantile_injections = np.array([np.mean(samples > 0) for samples in injections])
print(f'GW250114 median deltaA = {median_production:.3f}; A/A_GR = {1 - median_production:.2f}')
print(f'zero-noise median deltaA = {np.median(zero_noise):.3f}')
print(f'fraction with |median| >= GW250114 = {np.mean(np.abs(median_injections) >= abs(median_production)):.2f}')
print(f'fraction with GR quantile <= GW250114 = {np.mean(quantile_injections <= quantile_production):.2f}')

In [ ]:
x_limits = (-5, 1)
x_grid = np.linspace(*x_limits, 400)
fig, ax = plt.subplots()
for index, samples in enumerate(injections):
    ax.plot(x_grid, gaussian_kde(samples)(x_grid), ls='dashed', color='gray',
            lw=1.0, alpha=0.5, zorder=1,
            label='real-noise simulations' if index == 0 else None)

ax.hist(production, edgecolor='navy', facecolor='lightsteelblue', bins=50,
        density=True, histtype='stepfilled', lw=2, zorder=3, alpha=0.75,
        label='GW250114')
ax.hist(zero_noise, edgecolor='firebrick', facecolor='mistyrose', bins=50,
        density=True, histtype='stepfilled', lw=2, zorder=2, alpha=0.55,
        label='zero noise simulation')
for samples, color in ((production, 'navy'), (zero_noise, 'firebrick')):
    for value in np.percentile(samples, [5, 95]):
        ax.axvline(value, ls='dashed', color=color, lw=2, zorder=5)
ax.axvline(0, color='k', ls='dotted', lw=2, zorder=6, label='GR value')
ax.set_xlim(*x_limits)
ax.set_ylim(0, 2)
ax.set_xlabel(r'$\delta A_{22n\times22m}$')
ax.set_ylabel('Probability Density')
ax.legend(loc='upper left', frameon=False)

output = ROOT / 'figures/tgr_comparison_4Mf_exact_gaussian.pdf'
fig.savefig(output, bbox_inches='tight')
plt.show()